# Hierarchischer Ansatz: k-Nearest Neighbors

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** k-Nearest Neighbors je Stufe im hierarchischen Ansatz testen (#101).

In [ ]:
%load_ext autoreload
%autoreload 2

from sklearn.neighbors import KNeighborsClassifier

from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.data import load_split
from awp2.evaluation import compare_predictions
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import HierarchicalClassifier, summarize_hierarchical, summary_table
from awp2.models.tuned import refit_untracked, tuned_hierarchical_svm
from awp2.plots import plot_error_breakdown, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

k-Nearest Neighbors klassifiziert über die ähnlichsten Trainingsspektren. Mit Kosinus-Abstand zählt der Winkel zwischen Spektren – verwandt mit dem Spectral Angle Mapper der Fernerkundung. Frage: Reicht lokale Ähnlichkeit, oder braucht es eine gelernte Grenze wie bei der SVM? kNN kennt keine Klassengewichte; `weights="distance"` mildert das nur.

Gleicher Ablauf wie bei der Referenz `hierarchical_svm` (#87): CV-Suche mit `tune()` auf dem
Trainingsteil, dann genau ein `run()`. Verglichen wird per CV mit der SVM-Referenz, danach
Fehlerzerlegung, Oracle-Kultur und McNemar gegen die SVM auf der Validierung.

In [ ]:
reference = tuned_hierarchical_svm()
svm_cv_score = reference.cv_score
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
TOP_CANDIDATES = 15

model = HierarchicalClassifier(
    KNeighborsClassifier(weights="distance"), KNeighborsClassifier(weights="distance")
)

neighbors = [1, 3, 5, 9]
metrics = ["euclidean", "cosine"]
param_grid = {
    "crop_estimator__n_neighbors": neighbors,
    "crop_estimator__metric": metrics,
    "stage_estimator__n_neighbors": neighbors,
    "stage_estimator__metric": metrics,
}

## CV-Suche

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="hierarchical_knn_search",
        approach="hierarchical",
        description="kNN je Stufe im hierarchischen Ansatz (#101).",
        preprocessing=preprocessing,
        balance_samples=False,
        param_grid=param_grid,
    ),
)
tuned.best_params, round(tuned.best_score, 4), round(tuned.best_score - svm_cv_score, 4)

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="crop_estimator__metric"
)
tuning_ax.axvline(svm_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.set_title("Hierarchisch kNN: CV-Kandidaten (gestrichelt: SVM-Referenz)")
save_doc_figure(tuning_ax.figure, "hierarchical_knn_tuning", MODEL_DOCS_FIGURES_DIR)

## Einmalige Bewertung

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="hierarchical_knn",
        approach="hierarchical",
        description="Bester kNN aus hierarchical_knn_search, einmal bewertet.",
        preprocessing=preprocessing,
        balance_samples=False,
        tuning_run=tuned.run_id,
    ),
)
result.metrics.model_dump(), result.run_id

## Vergleich mit der SVM-Referenz

Die SVM-Referenz (`tuned_hierarchical_svm()`) wird ohne MLflow nachtrainiert
(`refit_untracked`), um die Vorhersagen Zeile für Zeile zu vergleichen.

In [ ]:
svm_result = refit_untracked(reference, "hierarchical_svm_refit")
summaries = {
    "RBF-SVM (Referenz)": summarize_hierarchical(svm_result.pipeline, split.X_val, split.y_val),
    "kNN": summarize_hierarchical(result.pipeline, split.X_val, split.y_val),
}
breakdowns = {label: summary.breakdown for label, summary in summaries.items()}
(
    summary_table(summaries).round(3),
    compare_predictions(split.y_val, svm_result.y_pred, result.y_pred),
)

In [ ]:
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung: kNN vs. RBF-SVM")
save_doc_figure(breakdown_ax.figure, "hierarchical_knn_breakdown", MODEL_DOCS_FIGURES_DIR)

## Befund

- 1 Nachbar mit Kosinus-Abstand auf beiden Stufen (Rasterrand bei k); Kosinus schlägt
  euklidisch.
- CV 0.742, Validierung 0.704 – der größte Abfall der Serie (hohe Varianz von 1-NN).
- Kultur falsch 14 %, und auch das Stadium ist schwach (mit wahrer Kultur 0.839).